# Training the EfficientNet-B4 Model on Google Colab (with Resume Training Support)

The file has been updated to support **resuming training** in case of internet disconnection or session interruption.

### How does it work?
- The code will save a `checkpoint.pth` file in Google Drive after every training cycle (Epoch).
- Upon restarting the code, it will automatically search for this file and resume from where it left off.

### Steps:
1. Ensure `Emotion data.zip` is uploaded to the root directory of your Google Drive.
2. Mount your drive and run the cells in order.

---

# تدريب نموذج EfficientNet-B4 على Google Colab (مع دعم استكمال التدريب)

تم تحديث الملف ليدعم **استكمال التدريب** في حال انقطاع الإنترنت أو الجلسة.

### كيف يعمل؟
- سيقوم الكود بحفظ ملف `checkpoint.pth` في Google Drive بعد كل دورة تدريبية (Epoch).
- عند إعادة تشغيل الكود، سيبحث تلقائياً عن هذا الملف ويكمل من حيث توقف.

### الخطوات:
1. تأكد من رفع `Emotion data.zip` إلى المجلد الرئيسي في Google Drive.
2. قم بربط الدرايف وتشغيل الخلايا بالترتيب.

---
هل ترغب في أن أساعدك في كتابة كود بايثون (PyTorch أو TensorFlow) الخاص بحفظ واسترجاع نقاط التوقف (Checkpoints) لهذا النموذج؟

In [ ]:
# 1. ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. فك ضغط البيانات (إذا لم تكن موجودة)
import os
if not os.path.exists("/content/dataset"):
    !unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset"
    print("تم فك ضغط البيانات.")
else:
    print("البيانات موجودة بالفعل.")

In [ ]:
# 3. تثبيت المكتبات
!pip install torch torchvision opencv-python-headless tqdm

In [ ]:
# 4. كود التدريب مع ميزة الحفظ التلقائي
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import time
from tqdm.notebook import tqdm

# مسارات الحفظ في الدرايف لضمان عدم ضياع الجهد
CHECKPOINT_PATH = "/content/drive/MyDrive/checkpoint.pth"
BEST_MODEL_PATH = "/content/drive/MyDrive/best_model_colab.pth"

# إعدادات البيانات
DATA_DIR = "/content/dataset/archive"
if not os.path.exists(os.path.join(DATA_DIR, 'train')):
    DATA_DIR = "/content/dataset"

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
BATCH_SIZE = 16
NUM_EPOCHS = 15 # يمكنك زيادة العدد
NUM_CLASSES = 7
IMG_SIZE = 380

def train_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"الجهاز المستخدم: {device}")

    # التحويلات
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        'test': transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

    image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x]) for x in ['train', 'test']}
    dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True if x=='train' else False, num_workers=2) for x in ['train', 'test']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
    
    # تحميل الموديل
    print("تحميل موديل EfficientNet-B4...")
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    start_epoch = 0
    best_acc = 0.0

    # --- فحص وجود نقطة استرجاع (Checkpoint) ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"==> تم العثور على نقطة استرجاع في الدرايف. جاري التحميل...")
        checkpoint = torch.load(CHECKPOINT_PATH)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_acc = checkpoint['best_acc']
        print(f"==> استكمال التدريب من الدورة (Epoch) رقم {start_epoch + 1}")

    for epoch in range(start_epoch, NUM_EPOCHS):
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
        print('-' * 10)

        for phase in ['train', 'test']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in tqdm(dataloaders[phase], desc=phase, leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), BEST_MODEL_PATH)
                print("تم حفظ أفضل موديل حتى الآن!")

        # --- حفظ نقطة استرجاع بعد كل دورة ---
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"Checkpoint saved (Epoch {epoch+1})")

if __name__ == '__main__':
    train_model()